# Ejercicios · Suficiencia energética del SIN 2026-2030

**Tutorial del Trabajo Final · Taller de Programación en Python (MECA 4110), 2026-2 · Universidad de los Andes**  
Carlos Andrés Terrassa Vallejo (código 200518213) · Esteban Darío Herrera León (código 202424968) · Jair Armando Castañeda Rodríguez (código 202616631)

Tres ejercicios sobre el tutorial `Tutorial_Suficiencia_SIN.ipynb`, de menor a mayor dificultad, con sus soluciones al final. La celda de abajo corre el tutorial completo, sin mostrar sus salidas, para tener sus tablas y funciones; tarda unos 10 segundos. El tutorial y su carpeta `datos_tutorial/` deben estar al lado de este notebook, como vienen en el .zip.

In [1]:
# Paquetes de este notebook; el tutorial carga los suyos al correr
from pathlib import Path

import requests
from IPython import get_ipython
from IPython.utils.capture import capture_output

TUTORIAL = Path("Tutorial_Suficiencia_SIN.ipynb")
if not TUTORIAL.exists():   # en Colab este notebook llega solo: el tutorial se baja del repositorio público
    respuesta = requests.get("https://raw.githubusercontent.com/cterrassa/suficiencia-sin/main/" + TUTORIAL.name, timeout=60)
    respuesta.raise_for_status()
    TUTORIAL.write_bytes(respuesta.content)
with capture_output():   # corre todas las celdas del tutorial en este mismo kernel, sin mostrar sus salidas
    get_ipython().run_line_magic("run", str(TUTORIAL))
print(f"Tutorial listo: {len(PERIODOS)} períodos, {len(ESCENARIOS)} escenarios y {len(CASOS)} casos del MILP")

Tutorial listo: 4 períodos, 3 escenarios y 8 casos del MILP


<a id="ejercicios" name="ejercicios"></a>
## Ejercicios para el lector

Usan las tablas y las funciones del tutorial, que la celda de arriba deja listas; las secciones que se citan son las del tutorial. Las soluciones están [al final](#soluciones).

**Los ejercicios se revisan solos.** Debajo de cada enunciado hay una celda para la respuesta: se reemplaza cada `None` por un número y se corre. La función `verificar_ejercicio_N()` calcula la respuesta esperada con las funciones del tutorial, la compara con la suya y dice qué está bien y qué no, con una pista, sin mostrar la solución. Las celdas de verificación revelan parte del método: conviene no leerlas antes de intentarlo.

| Función | Qué hace | Parámetros que usamos | Por qué así |
|---|---|---|---|
| `revisar(que, dado, esperado, tolerancia, pista)` (propia) | Compara una respuesta con el valor esperado e imprime ✓, ✗ con una pista, o · si todavía no hay respuesta | La tolerancia de cada respuesta | Una respuesta vacía o mal escrita no detiene el notebook |
| `verificar_ejercicio_1()`, `_2()` y `_3()` (propias) | Calculan las respuestas esperadas y llaman a `revisar()` | Los valores de la celda de respuesta | En el ejercicio 3 la tolerancia acepta otra semilla u otro número de simulaciones |

In [2]:
def revisar(que, dado, esperado, tolerancia, pista):
    """Compara una respuesta con el valor esperado e imprime ✓, ✗ con una pista, o · si falta la respuesta."""
    if dado is None:
        print(f"·  {que}: sin respuesta todavía")
        return
    try:
        correcto = abs(float(dado) - float(esperado)) <= tolerancia
    except (TypeError, ValueError):                   # texto u otro objeto en lugar de un número
        print(f"✗  {que}: la respuesta debe ser un número")
        return
    print(f"✓  {que}" if correcto else f"✗  {que}: no coincide. Pista: {pista}")

### Ejercicio 1 · Otras medidas de similitud *(fácil)*

Un ejercicio fácil: repetir la comparación de nombres de la sección 3.1 con otras medidas de RapidFuzz.

**Enunciado.** En la sección 3.1 se usaron `token_set_ratio` y `WRatio`. Compare cuatro medidas de RapidFuzz, `fuzz.ratio`, `fuzz.partial_ratio`, `fuzz.token_set_ratio` y `fuzz.WRatio`, en estos cinco pares:

| a | b | ¿Es la misma planta? |
|---|---|---|
| La Loma Solar | PARQUE SOLAR LA LOMA | Sí |
| La Loma Solar | SOLAR LA LOMA II | No: es una segunda etapa |
| Termo Tasajero II | TASAJERO 1 | No: son unidades distintas |
| Hidroituango | ITUANGO | Sí |
| Central Hidroeléctrica Guatapé | GUATAPE | Sí |

Normalice los nombres con `normalizar()` y marque qué medidas pasan el umbral elegido en la sección 3.1 (`elegido["umbral"]`). ¿Alguna medida acierta en los cinco pares? ¿Qué regla de la sección 3.1 corrige los errores que quedan?

**Pistas:**
- Arme una lista de diccionarios, una fila por par, y conviértala en `DataFrame`.
- `fuzz.ratio` compara las dos cadenas completas; `fuzz.partial_ratio` compara la cadena corta con el pedazo más parecido de la larga.
- `numeros()` devuelve los números de un nombre normalizado: dos etapas distintas tienen números distintos.

**Su respuesta:** cuántos de los cinco pares acierta cada medida, solo con el puntaje y sumando la prueba de números (un par se acepta si además los números coinciden).

In [3]:
def verificar_ejercicio_1(aciertos_solo, aciertos_con_numeros):
    """Revisa el ejercicio 1: aciertos de 5 de cada medida, solo con el puntaje y con la prueba de números."""
    pares = [("La Loma Solar", "PARQUE SOLAR LA LOMA", True), ("La Loma Solar", "SOLAR LA LOMA II", False),
             ("Termo Tasajero II", "TASAJERO 1", False), ("Hidroituango", "ITUANGO", True),
             ("Central Hidroeléctrica Guatapé", "GUATAPE", True)]
    medidas = {"ratio": fuzz.ratio, "partial_ratio": fuzz.partial_ratio,
               "token_set_ratio": fuzz.token_set_ratio, "WRatio": fuzz.WRatio}
    for m, f in medidas.items():
        acepta = [f(normalizar(a), normalizar(b)) >= elegido["umbral"] for a, b, _ in pares]
        iguales = [numeros(normalizar(a)) == numeros(normalizar(b)) for a, b, _ in pares]
        solo = sum(ac == misma for ac, (_, _, misma) in zip(acepta, pares))
        con_numeros = sum((ac and ig) == misma for ac, ig, (_, _, misma) in zip(acepta, iguales, pares))
        revisar(f"{m}, solo con el puntaje", (aciertos_solo or {}).get(m), solo, 0,
                f"normalice antes de puntuar y acepte con puntaje >= {elegido['umbral']:.0f}")
        revisar(f"{m}, con la prueba de números", (aciertos_con_numeros or {}).get(m), con_numeros, 0,
                "un par se acepta solo si pasa el umbral y sus números coinciden")

In [4]:
# Su respuesta al ejercicio 1: aciertos de 5 por medida (reemplace cada None por un número y corra la celda)
aciertos_solo = {"ratio": None, "partial_ratio": None, "token_set_ratio": None, "WRatio": None}
aciertos_con_numeros = {"ratio": None, "partial_ratio": None, "token_set_ratio": None, "WRatio": None}
verificar_ejercicio_1(aciertos_solo, aciertos_con_numeros)

·  ratio, solo con el puntaje: sin respuesta todavía
·  ratio, con la prueba de números: sin respuesta todavía
·  partial_ratio, solo con el puntaje: sin respuesta todavía
·  partial_ratio, con la prueba de números: sin respuesta todavía
·  token_set_ratio, solo con el puntaje: sin respuesta todavía
·  token_set_ratio, con la prueba de números: sin respuesta todavía
·  WRatio, solo con el puntaje: sin respuesta todavía
·  WRatio, con la prueba de números: sin respuesta todavía


### Ejercicio 2 · ¿Y si la obra de Ituango no llega? *(intermedio)*

Un ejercicio intermedio: medir cuánto depende la respuesta de un solo proyecto, combinando el balance de la sección 3.3 con el MILP de la 3.4.

**Enunciado.** La oferta firme de 2029-2030 incluye 2,17 GWh-día por la obra del embalse de Ituango: es uno de los proyectos de `nuevos` (sección 3.3). Suponga que la obra no llega:
1. Quite Ituango de `nuevos` y recalcule la OEF nueva con retrasos. ¿Cuántos GWh firmes se pierden en cada período?
2. Sume esa pérdida a la brecha del base y del estrés, y resuelva el MILP de cada escenario.
3. ¿Cambia el portafolio? ¿Cuánto cuestan los bloques en valor presente? Explique por qué la respuesta del base y la del estrés no son iguales.

**Pistas:**
- `oferta_nuevos(nuevos_sin_ituango, True)` da la OEF nueva con retrasos en GWh-día; réstela de la original y multiplíquela por `dias`.
- Con retrasos, solo una parte de la OEF de un proyecto entra en su período (sección 3.3), así que la pérdida no es 2,17 × 365.
- `correr(escenario, brecha_extra=perdida)` suma la pérdida a la brecha antes de resolver.
- Mire cuánto le sobra al portafolio del base en 2029-2030.

**Su respuesta:** los GWh firmes que se pierden en 2029-2030 y el costo de los bloques en valor presente (MUSD) del base y del estrés sin la obra.

In [5]:
def verificar_ejercicio_2(perdida_2029_2030, costo_base, costo_estres):
    """Revisa el ejercicio 2: GWh firmes que se pierden en 2029-2030 sin la obra de Ituango y costo de los
    bloques (MUSD en valor presente) del base y del estrés sin ella."""
    sin_obra = nuevos[~nuevos["nombre"].str.contains("ITUANGO", case=False, na=False)]
    perdida = (oferta_nuevos(nuevos, True) - oferta_nuevos(sin_obra, True)) * dias
    revisar("GWh que se pierden en 2029-2030", perdida_2029_2030, perdida["2029-2030"], 1,
            "con retrasos, en su primer período solo cuenta una parte de la OEF del proyecto")
    for nombre, dado, escenario in [("base", costo_base, "base"), ("estrés", costo_estres, "estres")]:
        esperado = correr(escenario, brecha_extra=perdida)[0]["costo_bloques_vp_musd"]
        revisar(f"costo de los bloques del {nombre} sin la obra", dado, esperado, 1,
                "sume la pérdida a la brecha con correr(escenario, brecha_extra=...) y lea costo_bloques_vp_musd")

In [6]:
# Su respuesta al ejercicio 2 (reemplace cada None por un número y corra la celda)
perdida_2029_2030 = None       # GWh firmes que se pierden en 2029-2030
costo_base_sin_obra = None     # costo de los bloques del base sin la obra, MUSD en valor presente
costo_estres_sin_obra = None   # lo mismo, en el estrés
verificar_ejercicio_2(perdida_2029_2030, costo_base_sin_obra, costo_estres_sin_obra)

·  GWh que se pierden en 2029-2030: sin respuesta todavía
·  costo de los bloques del base sin la obra: sin respuesta todavía
·  costo de los bloques del estrés sin la obra: sin respuesta todavía


### Ejercicio 3 · ¿Nueve plantas o tres? *(difícil)*

Un ejercicio difícil, que extiende la sección 5.4: cuánto cambia la incertidumbre si la muestra de retrasos de 2019 vale por menos plantas de las que parece.

**Enunciado.** La mejora 1 de la sección 5.4 sortea las probabilidades de entrada como si vinieran de nueve plantas iguales. Pero no pesan lo mismo: Tesorito tiene más de la mitad de la OEF de las nueve plantas nuevas de la subasta de 2019. El **tamaño efectivo de Kish**, $n_{ef} = (\sum_i w_i)^2 / \sum_i w_i^2$, con $w_i$ la OEF de cada planta, dice a cuántas plantas iguales equivale la muestra.
1. Calcule $n_{ef}$ con la OEF de las nueve plantas.
2. Repita la simulación de la sección 5.4 con $n_{ef}$ plantas en lugar de nueve: primero sin la incertidumbre de la demanda y después con ella. Compare el P80 de 2029-2030 con el de nueve plantas.
3. ¿Por qué la diferencia entre nueve plantas y $n_{ef}$ es menor cuando se incluye la demanda?

**Pistas:**
- `s2019["oef_registrada_gwh_dia"]` (sección 3.1) trae la OEF de las nueve plantas.
- `simular_brecha(tamano_muestra=..., demanda_incierta=False)` deja fija la demanda en el escenario medio.
- El P80 de 2029-2030 es `sim["2029-2030"].quantile(0.8)`.
- Para la pregunta 3, compare cuánto se dispersa la brecha por los retrasos y cuánto por la demanda.

**Su respuesta:** $n_{ef}$ y el P80 de la brecha de 2029-2030 con $n_{ef}$ plantas, sin y con la incertidumbre de la demanda (GWh). Como el resultado es aleatorio, la revisión acepta otra semilla.

In [7]:
def verificar_ejercicio_3(n_ef, p80_sin_demanda, p80_con_demanda):
    """Revisa el ejercicio 3: tamaño efectivo de Kish de las nueve plantas de 2019 y el P80 de la brecha de
    2029-2030 con ese tamaño, sin y con la incertidumbre de la demanda (referencia de 50.000 simulaciones)."""
    w = s2019["oef_registrada_gwh_dia"]
    esperado = w.sum() ** 2 / (w ** 2).sum()
    revisar("tamaño efectivo de Kish", n_ef, esperado, 0.01, "(suma de la OEF)² dividido por la suma de la OEF al cuadrado")
    for que, dado, incierta, tolerancia in [("P80 de 2029-2030 sin la demanda", p80_sin_demanda, False, 120),
                                            ("P80 de 2029-2030 con la demanda", p80_con_demanda, True, 250)]:
        referencia = simular_brecha(50_000, SEMILLA + 1, tamano_muestra=esperado, demanda_incierta=incierta)
        revisar(que, dado, referencia["2029-2030"].quantile(0.8), tolerancia,
                "simular_brecha(tamano_muestra=n_ef, demanda_incierta=...) y el cuantil 0,8 de 2029-2030")

In [8]:
# Su respuesta al ejercicio 3 (reemplace cada None por un número y corra la celda)
n_ef = None              # tamaño efectivo de Kish de las nueve plantas de 2019
p80_sin_demanda = None   # P80 de la brecha de 2029-2030 con n_ef plantas y la demanda media, GWh
p80_con_demanda = None   # lo mismo, con la incertidumbre de la demanda, GWh
verificar_ejercicio_3(n_ef, p80_sin_demanda, p80_con_demanda)

·  tamaño efectivo de Kish: sin respuesta todavía


·  P80 de 2029-2030 sin la demanda: sin respuesta todavía

·  P80 de 2029-2030 con la demanda: sin respuesta todavía


<a id="soluciones" name="soluciones"></a>
## Soluciones de los ejercicios

Cada solución trae su código, la revisión automática que la confirma y una lectura del resultado.

### Solución 1

Se normalizan los cinco pares, se puntúan con las cuatro medidas y se cuenta cuántos acierta cada una, con y sin la prueba de números.

In [9]:
# Solución 1: cuatro medidas de RapidFuzz sobre cinco pares, después de normalizar
umbral = elegido["umbral"]
pares = [("La Loma Solar", "PARQUE SOLAR LA LOMA", True), ("La Loma Solar", "SOLAR LA LOMA II", False),
         ("Termo Tasajero II", "TASAJERO 1", False), ("Hidroituango", "ITUANGO", True),
         ("Central Hidroeléctrica Guatapé", "GUATAPE", True)]
medidas = {"ratio": fuzz.ratio, "partial_ratio": fuzz.partial_ratio,
           "token_set_ratio": fuzz.token_set_ratio, "WRatio": fuzz.WRatio}

filas = []
for a, b, misma in pares:
    na, nb = normalizar(a), normalizar(b)
    fila = {"a": a, "b": b, "normalizados": f"{na} | {nb}", "misma planta": misma,
            "números iguales": numeros(na) == numeros(nb)}
    fila.update({m: round(f(na, nb), 1) for m, f in medidas.items()})
    filas.append(fila)
sol1 = pd.DataFrame(filas)
display(sol1)

# ¿Acierta cada medida? Solo con el puntaje (>= umbral) y sumando la prueba de números
acepta = sol1[list(medidas)].ge(umbral)
aciertos = pd.DataFrame({
    "solo con el puntaje": acepta.eq(sol1["misma planta"], axis=0).sum(),
    "con la prueba de números": (acepta & sol1[["números iguales"]].to_numpy()).eq(sol1["misma planta"], axis=0).sum()})
aciertos.rename_axis(f"aciertos de 5 (umbral {umbral:.0f})")

,a,b,normalizados,misma planta,números iguales,ratio,partial_ratio,token_set_ratio,WRatio
0,La Loma Solar,PARQUE SOLAR LA LOMA,LOMA | LOMA,True,True,100,100,100,100
1,La Loma Solar,SOLAR LA LOMA II,LOMA | LOMA 2,False,False,80,100,100,90
2,Termo Tasajero II,TASAJERO 1,TERMO TASAJERO 2 | TASAJERO 1,False,False,"69,2",90,"88,9","85,5"
3,Hidroituango,ITUANGO,HIDROITUANGO | ITUANGO,True,True,"73,7",100,"73,7",90
4,Central Hidroeléctrica Guatapé,GUATAPE,GUATAPE | GUATAPE,True,True,100,100,100,100


,solo con el puntaje,con la prueba de números
aciertos de 5 (umbral 86),,
ratio,4,4
partial_ratio,3,5
token_set_ratio,2,4
WRatio,4,5


In [10]:
# La solución pasa la revisión del ejercicio 1
verificar_ejercicio_1(aciertos["solo con el puntaje"].to_dict(), aciertos["con la prueba de números"].to_dict())

✓  ratio, solo con el puntaje
✓  ratio, con la prueba de números
✓  partial_ratio, solo con el puntaje
✓  partial_ratio, con la prueba de números
✓  token_set_ratio, solo con el puntaje
✓  token_set_ratio, con la prueba de números
✓  WRatio, solo con el puntaje
✓  WRatio, con la prueba de números


**Lectura.** Solo con el puntaje, ninguna medida acierta en los cinco pares: `ratio` y `WRatio` fallan en uno, `partial_ratio` en dos y `token_set_ratio`, la medida que escogió la sección 3.1, en tres. Las dos trampas son de número: LOMA frente a LOMA 2 y TASAJERO 2 frente a TASAJERO 1 se parecen casi del todo. Con la prueba de números (la regla `numeros_ok` de la sección 3.1, aquí simplificada, porque la del tutorial también acepta un 1 de un solo lado), `partial_ratio` y `WRatio` aciertan los cinco. Hidroituango es el caso contrario: es la misma planta, con el prefijo HIDRO pegado al nombre, y `ratio` y `token_set_ratio` lo rechazan porque comparan cadenas o palabras completas.

Cinco pares no bastan para escoger una medida. La sección 3.1 la escoge con 205 pares revisados, donde lo que pesa son los cientos de autogeneradores con nombres parecidos a los de los proyectos. Una extensión natural es agregar `partial_ratio` a esa curva.

### Solución 2

Se quita la obra de Ituango de los proyectos nuevos, se mide la OEF que se pierde y se resuelve el MILP con esa pérdida sumada a la brecha.

In [11]:
# Solución 2: la OEF que se pierde sin la obra de Ituango y el portafolio con y sin ella
sin_ituango = nuevos[~nuevos["nombre"].str.contains("ITUANGO", case=False, na=False)]
perdida = (oferta_nuevos(nuevos, True) - oferta_nuevos(sin_ituango, True)) * dias
print("GWh firmes que se pierden sin la obra:", ", ".join(f"{p}: {es(v, 1)}" for p, v in perdida.items()))

filas = {}
for escenario in ["base", "estres"]:
    for caso, extra in [("con la obra", 0), ("sin la obra", perdida)]:
        resumen, _ = correr(escenario, brecha_extra=extra)
        mw = {NOMBRES_TECNOLOGIA[k[3:]]: v for k, v in resumen.items() if k.startswith("mw_") and v > 0}
        filas[(escenario, caso)] = mw | {"costo de los bloques (MUSD, VP)": round(resumen["costo_bloques_vp_musd"], 1)}
pd.DataFrame(filas).fillna(0).T.rename_axis(["escenario", "Ituango"])

GWh firmes que se pierden sin la obra: 2026-2027: 0,0, 2027-2028: 0,0, 2028-2029: 0,0, 2029-2030: 420,3


Bagazo  Gas, ciclo abierto  Solar FV  costo de los bloques (MUSD, VP)  Eólica
escenario Ituango                                                                                   
base      con la obra      40                 600       200                            422,9       0
          sin la obra      40                 600       200                            422,9       0
estres    con la obra      40               1.200       200                            809,9     100
          sin la obra      40               1.300       100                            818,7       0

In [12]:
# La solución pasa la revisión del ejercicio 2
verificar_ejercicio_2(perdida["2029-2030"], filas[("base", "sin la obra")]["costo de los bloques (MUSD, VP)"],
                      filas[("estres", "sin la obra")]["costo de los bloques (MUSD, VP)"])

✓  GWh que se pierden en 2029-2030


✓  costo de los bloques del base sin la obra


✓  costo de los bloques del estrés sin la obra


**Lectura.** Sin la obra se pierden 420 GWh firmes en 2029-2030: con retrasos solo cuenta el 53% de los 2,17 GWh-día de Ituango (0,53 × 2,17 × 365). En el base, el portafolio y su costo no cambian (600 MW de ciclo abierto, 200 solares y 40 de bagazo, por 423 MUSD): lo que se construye en 2028-2029 para una brecha de 5.168 GWh sobra en 2029-2030, cuando la brecha sube de 4.585 a 5.005 GWh. En el estrés no hay esa holgura: la brecha de 2029-2030 pasa de 9.933 a 10.354 GWh, y el MILP cambia los 100 MW eólicos y un bloque solar de 100 MW por un bloque más de ciclo abierto (1.300 MW en total). El costo de los bloques sube de 810 a 819 MUSD, un 1,1%. La misma pérdida cambia el portafolio o no según cuánto le sobre al período, porque el MILP construye con bloques indivisibles.

### Solución 3

Se calcula el tamaño efectivo de Kish de las nueve plantas y se repite la simulación de la sección 5.4 con ese tamaño, sin y con la incertidumbre de la demanda.

In [13]:
# Solución 3: el tamaño efectivo de la muestra de 2019 y su efecto sobre el P80 de 2029-2030
w = s2019["oef_registrada_gwh_dia"]
n_ef = w.sum() ** 2 / (w ** 2).sum()
print(f"Tamaño efectivo de Kish: {es(n_ef, 2)} plantas iguales (Tesorito tiene el {es(100 * w.max() / w.sum())}% de la OEF)")

filas = {"9 plantas (sección 5.4)": 9, "plantas efectivas (Kish)": n_ef}
p80 = pd.DataFrame({demanda_txt: {fila: simular_brecha(tamano_muestra=tamano, demanda_incierta=incierta)["2029-2030"]
                                  .quantile(0.8) for fila, tamano in filas.items()}
                    for demanda_txt, incierta in [("sin la demanda", False), ("con la demanda", True)]})
p80.loc["diferencia"] = p80.loc["plantas efectivas (Kish)"] - p80.loc["9 plantas (sección 5.4)"]
p80.round(0).rename_axis("P80 de la brecha de 2029-2030 (GWh)")

Tamaño efectivo de Kish: 3,19 plantas iguales (Tesorito tiene el 53% de la OEF)


,sin la demanda,con la demanda
P80 de la brecha de 2029-2030 (GWh),,
9 plantas (sección 5.4),5.737,9.427
plantas efectivas (Kish),6.131,9.675
diferencia,394,248


In [14]:
# La solución pasa la revisión del ejercicio 3
kish = p80.loc["plantas efectivas (Kish)"]
verificar_ejercicio_3(n_ef, kish["sin la demanda"], kish["con la demanda"])

✓  tamaño efectivo de Kish


✓  P80 de 2029-2030 sin la demanda


✓  P80 de 2029-2030 con la demanda


**Lectura.** Las nueve plantas valen por 3,19: Tesorito tiene el 53% de la OEF. Con esa muestra efectiva, las probabilidades sorteadas se dispersan más, y el P80 de la brecha de 2029-2030 sube 394 GWh sin la incertidumbre de la demanda (de 5.737 a 6.131), un 7%. Con la demanda, la diferencia es menor: 248 GWh (de 9.427 a 9.675), un 3%.

La razón es que la demanda dispersa la brecha mucho más que los retrasos: el escenario alto de la UPME queda 5.348 GWh por encima del medio en 2029-2030, mientras que medir mejor los retrasos mueve el P80 unos cientos de GWh. Cuando se suman dos fuentes de incertidumbre, la más ancha domina los percentiles. Para la respuesta, eso quiere decir que el plan del P80 depende sobre todo de cuánto se crea en la proyección de la UPME.